# Visualize đầu ra — Adaptive PB-JCI Online

Hai figure:
1. **PanNuke (test)** — ảnh từ **nửa TEST của Fold 3** (q̂ fit trên nửa CAL, cùng seed bảng kết quả). 5 lớp → 5 dòng + overlay nhân.
2. **CoNSeP (cross-dataset)** — từ `consep_preds.pkl`, **tự dò cấu trúc** và hiện luôn.

In [ ]:
%%writefile viz_pbjci.py
from __future__ import annotations
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

PANNUKE_CLASSES = ["Tan sinh (u)", "Viem", "Mo lien ket", "Chet", "Bieu mo"]
CLASS_COLORS = ["#e41a1c", "#4daf4a", "#377eb8", "#000000", "#ff7f00",
                "#984ea3", "#a65628", "#999999"]

def empirical_quantile(scores, alpha=0.1):
    n = len(scores)
    if n == 0: return float("inf")
    level = np.ceil((n + 1) * (1 - alpha)) / n
    return float(np.quantile(scores, min(level, 1.0), method="higher"))

def pb_count(scores, probs): return (scores[:, None] * probs).sum(axis=0)
def pb_variance(scores, probs):
    w = scores[:, None] * probs; return (w * (1.0 - w)).sum(axis=0)

def joint_scores(preds, gts):
    S = []
    for pred, gt in zip(preds, gts):
        s, p = np.asarray(pred["scores"]), np.asarray(pred["probs"])
        gt = np.asarray(gt, dtype=float); K = len(gt)
        if len(s) == 0: S.append(max(abs(gt[k]) for k in range(K)))
        else:
            E = pb_count(s, p); sg = np.sqrt(pb_variance(s, p) + 1e-6)
            S.append(max(abs(gt[k] - E[k]) / sg[k] for k in range(K)))
    return np.array(S)

def fit_qhat(cal_preds, cal_gts, alpha=0.1):
    return empirical_quantile(joint_scores(cal_preds, cal_gts), alpha)

def split_cal_test(n, cal_ratio=0.5, seed=42):
    rng = np.random.RandomState(seed); idx = rng.permutation(n)
    return idx[:int(n * cal_ratio)], idx[int(n * cal_ratio):]

def counts_to_interval(pred, q_hat):
    s, p = np.asarray(pred["scores"]), np.asarray(pred["probs"])
    if len(s) == 0:
        z = np.zeros(p.shape[1] if p.ndim == 2 else 5); return z, z, z, z
    E = pb_count(s, p); sg = np.sqrt(pb_variance(s, p) + 1e-6)
    return E, sg, np.maximum(0, E - q_hat * sg), E + q_hat * sg

def _instance_centroids(channel):
    ids = np.unique(channel); ids = ids[ids > 0]; cents = []
    for i in ids:
        ys, xs = np.where(channel == i); cents.append((ys.mean(), xs.mean()))
    return cents

def overlay_nuclei(image, masks_per_type, ax, names):
    ax.imshow(image)
    for k, ch in enumerate(masks_per_type):
        cents = _instance_centroids(ch)
        if cents:
            ys, xs = zip(*cents)
            ax.scatter(xs, ys, s=42, c=CLASS_COLORS[k], edgecolors="white",
                       linewidths=0.8, label=f"{names[k]} ({len(cents)})", zorder=3)
    ax.set_title("Anh + tam nhan (mau theo loai)"); ax.axis("off")
    # chu thich DUOI anh (ngang) -> khong che goc anh
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=3,
              fontsize=8, framealpha=0.9, title="Loai (so dem that)")

def _interval_panel(ax, E, lower, upper, gt, names, title):
    K = len(E); covered = (gt >= lower) & (gt <= upper); y = np.arange(K)[::-1]
    umax = max(float(np.max(upper)), 1.0)
    for i, yi in zip(range(K), y):
        c = "#2ca02c" if covered[i] else "#d62728"
        ax.plot([lower[i], upper[i]], [yi, yi], color=c, lw=7, alpha=0.35, solid_capstyle="round")
        ax.plot(E[i], yi, "o", color=c, ms=9)
        ax.plot(gt[i], yi, "D", color="black", ms=8, mfc="white", mew=1.6)
        ax.text(upper[i] + 0.012 * umax, yi, f"[{lower[i]:.0f}, {upper[i]:.0f}]", va="center", fontsize=9, color=c)
    ax.set_xlim(-0.02 * umax, umax * 1.42)   # chua chat de text [..] khong tran ra ngoai
    ax.set_yticks(y); labs = ax.set_yticklabels(names)
    for i, lab in zip(range(K), labs):
        lab.set_color(CLASS_COLORS[i]); lab.set_fontweight("bold")
    ax.set_xlabel("So te bao")
    cov_all = "DUNG ca lop" if covered.all() else "co lop TRUOT"
    ax.set_title(f"{title}\n(joint coverage: {cov_all})")
    ax.legend(handles=[Line2D([0],[0],marker="o",color="gray",ls="",ms=9,label="so doan E[N]"),
                       Line2D([0],[0],marker="D",color="black",ls="",ms=8,mfc="white",label="that")],
              loc="lower right", fontsize=9)
    ax.grid(axis="x", alpha=0.25); ax.margins(y=0.15); return covered

def visualize_prediction(pred, gt, q_hat, image=None, masks_per_type=None,
                         class_names=None, title="Dau ra 1 anh", save_path="fig.png"):
    gt = np.asarray(gt, dtype=float); K = len(gt)
    names = class_names or PANNUKE_CLASSES[:K]
    E, _, lower, upper = counts_to_interval(pred, q_hat)
    has_left = image is not None
    fig, axes = plt.subplots(1, 2 if has_left else 1, figsize=(13 if has_left else 7, 5.2))
    axes = np.atleast_1d(axes)
    if has_left:
        if masks_per_type is not None: overlay_nuclei(image, masks_per_type, axes[0], names)
        else: axes[0].imshow(image); axes[0].set_title("Anh"); axes[0].axis("off")
    covered = _interval_panel(axes[-1], E, lower, upper, gt, names, title)
    fig.text(0.5, -0.02, f"Tong: doan {E.sum():.1f} | that {gt.sum():.0f}"
             f"  -- dau ra chinh la {K} khoang theo loai", ha="center", fontsize=9, style="italic")
    plt.tight_layout(); fig.savefig(save_path, dpi=140, bbox_inches="tight"); plt.close(fig)
    print("Da luu:", save_path, "| covered:", covered.tolist()); return covered

# ---------- TU DO cau truc pkl ----------
SCORE_KEYS = ["scores","score","conf","confidence","objectness","det_scores","s"]
PROB_KEYS  = ["probs","prob","cls_probs","class_probs","probabilities","type_probs","p"]
GT_KEYS    = ["gt","gts","counts","gt_counts","labels","label","y","target","count"]
PREDLIST_KEYS = ["preds","predictions","pred"]
IMG_KEYS   = ["images","imgs","image","img"]
MSK_KEYS   = ["masks","mask","inst_map","instance_maps"]

def _get(d, keys):
    if isinstance(d, dict):
        for k in keys:
            if k in d: return d[k]
    return None

def _norm_pred(d):
    s = _get(d, SCORE_KEYS); p = _get(d, PROB_KEYS)
    s = np.asarray(s).ravel() if s is not None else np.array([])
    p = np.asarray(p) if p is not None else np.zeros((0, 1))
    return {"scores": s, "probs": p}

def auto_extract(obj):
    """Tu nhan dien -> (preds, gts, imgs, msks). Hoat dong voi dict-of-lists hoac list-of-dicts."""
    imgs = _get(obj, IMG_KEYS); msks = _get(obj, MSK_KEYS)
    if isinstance(obj, dict):
        plist = _get(obj, PREDLIST_KEYS)
        gts   = _get(obj, GT_KEYS)
        if plist is not None:
            preds = [_norm_pred(d) for d in plist]
            if gts is None:
                gts = [np.asarray(_get(d, GT_KEYS)) for d in plist]
            return preds, [np.asarray(g, float) for g in gts], imgs, msks
        s_all = _get(obj, SCORE_KEYS); p_all = _get(obj, PROB_KEYS)
        if s_all is not None and p_all is not None:
            preds = [{"scores": np.asarray(s).ravel(), "probs": np.asarray(p)}
                     for s, p in zip(s_all, p_all)]
            return preds, [np.asarray(g, float) for g in gts], imgs, msks
        raise ValueError("Khong nhan dien duoc dict; keys=" + str(list(obj.keys())))
    preds = [_norm_pred(d) for d in obj]
    gts   = [np.asarray(_get(d, GT_KEYS), float) for d in obj]
    return preds, gts, None, None


In [ ]:
# Cell 3 — Config
import numpy as np, glob, os, pickle
from IPython.display import Image as IPImage, display

SEED = 42; ALPHA = 0.1; FOLD = 3
PANNUKE_PREDS_PKL = "/kaggle/input/datasets/hipinhththu/pathosam-preds/pathosam_predictions.pkl"
PANNUKE_SETTING   = "in_dist"   # in_dist | mild_shift | severe_shift
CONSEP_PREDS_PKL  = "/kaggle/input/datasets/hipinhththu/consep-preds/consep_preds.pkl"

base = f"/kaggle/input/datasets/hipinhththu/pannuke/fold{FOLD}/Fold {FOLD}"
img_path = f"{base}/images/fold{FOLD}/images.npy"
msk_path = f"{base}/masks/fold{FOLD}/masks.npy"
if not os.path.exists(msk_path):
    msk_path = glob.glob(f"/kaggle/input/**/fold{FOLD}/masks.npy", recursive=True)[0]
    img_path = glob.glob(f"/kaggle/input/**/fold{FOLD}/images.npy", recursive=True)[0]
images = np.load(img_path, mmap_mode="r"); masks = np.load(msk_path, mmap_mode="r")
print("PanNuke:", images.shape, masks.shape)

def counts_from_mask(m6):
    per = np.asarray(m6[..., :5], dtype=np.int32)
    return np.array([int(np.unique(per[..., k]).size - 1) for k in range(5)])
def masks_per_type(m6):
    return np.asarray(m6[..., :5], dtype=np.int32).transpose(2, 0, 1)


In [ ]:
# Cell 4 — PanNuke: dung preds THAT (pathosam_predictions.pkl) -> hien
import importlib, viz_pbjci; importlib.reload(viz_pbjci)
from viz_pbjci import fit_qhat, split_cal_test, visualize_prediction, auto_extract, counts_to_interval

with open(PANNUKE_PREDS_PKL, "rb") as f: obj = pickle.load(f)

if isinstance(obj, dict) and "predictions_by_setting" in obj:
    pbs = obj["predictions_by_setting"]
    setting = PANNUKE_SETTING if PANNUKE_SETTING in pbs else list(pbs.keys())[0]
    if setting != PANNUKE_SETTING:
        print(f"[!] '{PANNUKE_SETTING}' khong co; dung '{setting}'. Cac setting: {list(pbs.keys())}")
    preds_all = list(pbs[setting])
    gts_pred  = np.asarray(obj["gt_counts"], dtype=float)
    idx_map   = np.asarray(obj["indices"]) if "indices" in obj else np.arange(len(preds_all))
else:
    setting = "(mask-gt)"
    preds_all, _, _, _ = auto_extract(obj)
    gts_pred = np.array([counts_from_mask(masks[i]) for i in range(len(preds_all))], dtype=float)
    idx_map = np.arange(len(preds_all))

M = len(preds_all)
assert len(gts_pred) == M, f"gt_counts {len(gts_pred)} != preds {M}"
print(f"PanNuke preds THAT: {M} anh | setting={setting} | gt_counts{gts_pred.shape}")

cal_idx, test_idx = split_cal_test(M, 0.5, seed=SEED)
q_hat = fit_qhat([preds_all[i] for i in cal_idx], [gts_pred[i] for i in cal_idx], ALPHA)

def _cov_i(i):
    E,_,lo,hi = counts_to_interval(preds_all[i], q_hat); return bool(((gts_pred[i]>=lo)&(gts_pred[i]<=hi)).all())
print(f"q_hat (CAL) = {round(q_hat,3)} | coverage test = {round(np.mean([_cov_i(i) for i in test_idx]),3)}")

# chon anh NHIEU LOAI nhat -> trong do uu tien PHU DUNG -> gan trung vi mat do
nnz = np.array([np.count_nonzero(gts_pred[i]) for i in test_idx])
sub = [test_idx[k] for k in range(len(test_idx)) if nnz[k] == nnz.max()]
cov_sub = [i for i in sub if _cov_i(i)]
pool = cov_sub if cov_sub else sub
tot = np.array([gts_pred[i].sum() for i in pool]); j = int(pool[np.argmin(np.abs(tot - np.median(tot)))])
print(f"Chon anh: pred-idx {j} | so loai={int(nnz.max())} | gt={gts_pred[j].astype(int)} | covered={_cov_i(j)}")

di = int(idx_map[j])
gt_mask = counts_from_mask(masks[di])
if not np.array_equal(gt_mask, gts_pred[j].astype(int)):
    print(f"[!] gt_counts {gts_pred[j].astype(int)} != mask {gt_mask} (dataset-idx {di}) -> dung gt MASK cho khop overlay")
    gt_show = gt_mask.astype(float)
else:
    gt_show = gts_pred[j]

img_j = np.asarray(images[di]); img_j = (img_j*255 if img_j.max() <= 1 else img_j).astype("uint8")
visualize_prediction(preds_all[j], gt_show, q_hat, image=img_j, masks_per_type=masks_per_type(masks[di]),
                     title=f"PanNuke PathoSAM TEST #{j} (setting={setting}, K=5) - THAT",
                     save_path="/kaggle/working/fig_pannuke.png")
display(IPImage("/kaggle/working/fig_pannuke.png"))


In [ ]:
# Cell 5 — CoNSeP: TU DO cau truc + hien luon (khong can chinh tay)
import importlib, viz_pbjci; importlib.reload(viz_pbjci)  # nap lai ban moi nhat (tranh cache cu)
from viz_pbjci import auto_extract, fit_qhat, split_cal_test, visualize_prediction

with open(CONSEP_PREDS_PKL, "rb") as f: consep = pickle.load(f)
print("type:", type(consep).__name__)
if isinstance(consep, dict):
    for k, v in consep.items():
        print(f"  key={k!r:18s} {type(v).__name__} {'len='+str(len(v)) if hasattr(v,'__len__') else ''}")
elif isinstance(consep, (list, tuple)):
    print("  list len", len(consep), "| elem0 keys:",
          list(consep[0].keys()) if isinstance(consep[0], dict) else type(consep[0]).__name__)

c_preds, c_gts, c_imgs, c_msks = auto_extract(consep)
c_gts = [np.asarray(g, dtype=float) for g in c_gts]
K = len(c_gts[0])
print(f"-> Tu dong nhan: {len(c_preds)} anh, K={K} lop; "
      f"pred[0]: scores{np.asarray(c_preds[0]['scores']).shape} probs{np.asarray(c_preds[0]['probs']).shape}; gt[0]={c_gts[0]}")


In [ ]:
# Cell 6 — CoNSeP: fit q_hat + chon patch DIEN HINH + nap ANH patch (cat tu Overlay)
import re
from PIL import Image as PILImage
from viz_pbjci import counts_to_interval

CONSEP_ROOT = "/kaggle/input/datasets/hipinhththu/consep-dataset/CoNSeP"
c_src = consep.get("src") if isinstance(consep, dict) else None

cal_i, test_i = split_cal_test(len(c_preds), 0.5, seed=SEED)
q_consep = fit_qhat([c_preds[i] for i in cal_i], [c_gts[i] for i in cal_i], ALPHA)
def _cov_i(i):
    E,_,lo,hi = counts_to_interval(c_preds[i], q_consep); return bool(((c_gts[i]>=lo)&(c_gts[i]<=hi)).all())
print(f"q_hat CoNSeP (CAL) = {round(q_consep,3)} | coverage test = {round(np.mean([_cov_i(i) for i in test_i]),3)}")

# uu tien patch 'test_*' (chac chan co trong Test/Overlay) -> phu dung -> gan trung vi
base = [i for i in test_i if (c_src is not None and str(c_src[i]).startswith("test_"))] or list(test_i)
pool = [i for i in base if _cov_i(i)] or base
tot = np.array([c_gts[i].sum() for i in pool]); jc = int(pool[np.argmin(np.abs(tot - np.median(tot)))])

# nap + cat anh patch tu Overlay
img_c = None
if c_src is not None:
    s = str(c_src[jc]); parts = s.split("_")
    img_name = f"{parts[0]}_{parts[1]}"; folder = "Train" if parts[0] == "train" else "Test"
    m = re.match(r"r(\d+)c(\d+)", parts[2]); r, c = int(m.group(1)), int(m.group(2))
    p = f"{CONSEP_ROOT}/{folder}/Overlay/{img_name}.png"
    if os.path.exists(p):
        full = np.asarray(PILImage.open(p).convert("RGB"))
        rc = [re.match(r".*_r(\d+)c(\d+)$", x) for x in c_src if str(x).startswith(img_name + "_")]
        nr = max(int(z.group(1)) for z in rc) + 1; nc = max(int(z.group(2)) for z in rc) + 1
        H, W = full.shape[:2]; ph, pw = H // nr, W // nc
        img_c = full[r*ph:(r+1)*ph, c*pw:(c+1)*pw]
        print(f"CoNSeP patch '{s}' -> {folder}/Overlay/{img_name}.png | grid {nr}x{nc} | crop {img_c.shape[:2]}")
    else:
        print(f"[!] Khong thay {p} -> khong overlay")
else:
    print("[i] pkl khong co 'src' -> khong overlay")

names_consep = (["Bieu mo","Viem","Spindle","Khac"][:K] if K == 4 else (["Tong"] if K == 1 else None))
visualize_prediction(c_preds[jc], c_gts[jc], q_consep, image=img_c, masks_per_type=None,
                     class_names=names_consep, title=f"CoNSeP patch #{jc} ({s if c_src is not None else ''}, cross, K={K})",
                     save_path="/kaggle/working/fig_consep.png")
display(IPImage("/kaggle/working/fig_consep.png"))
